# U.S. County Population by Census Year via IPUMS NHGIS API

This notebook retrieves historical total-population counts for every U.S. county
from the [IPUMS NHGIS](https://www.nhgis.org/) data collection using the
[IPUMS API](https://developer.ipums.org/docs/v2/apiprogram/) and the
[`ipumspy`](https://ipumspy.readthedocs.io/) Python client.

**What the notebook does:**

1. Connects to the NHGIS API and inspects metadata for the time-series table we need.
2. Submits an extract request for county-level total population across all decennial censuses.
3. Downloads the resulting data and loads it into a Pandas DataFrame.
4. Reshapes the data into a wide table: one row per county, one column per census year.
5. Exports the table to CSV.
6. Produces a time-series chart of the aggregate U.S. population.

### Prerequisites

- An [IPUMS account](https://www.nhgis.org/) registered for NHGIS access.
- An API key from <https://account.ipums.org/api_keys>.
- The key stored in the environment variable `IPUMS_API_KEY`.

```bash
pip install ipumspy pandas matplotlib
export IPUMS_API_KEY="your-key-here"
```

### Data source

We use **NHGIS Time Series Table A00** (*Persons: Total*) with **nominal**
geographic integration. This table links comparable total-population counts
across decennial censuses (1790–2020) at the county level. Under nominal
integration each row maps to a county as defined at the time of each census,
and a cell is non-empty only for censuses in which that county existed.

---
## 1. Imports and configuration

In [ ]:
import os
import re
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from ipumspy import (
    IpumsApiClient,
    AggregateDataExtract,
    TimeSeriesTable,
)
from ipumspy.api.metadata import TimeSeriesTableMetadata

# Directories for downloaded data and outputs
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZIP_PATH = DATA_DIR / "nhgis_extract.zip"
CSV_OUTPUT = DATA_DIR / "county_population_by_census_year.csv"

---
## 2. Connect to the IPUMS API

We read the API key from the `IPUMS_API_KEY` environment variable and
create an `IpumsApiClient` instance, which handles authentication,
retries, and rate limiting.

In [2]:
api_key = "59cba10d8a5da536fc06b59df1d9eca2d6114582ac1c002722d948da"
if not api_key:
    raise RuntimeError(
        "Please set the IPUMS_API_KEY environment variable.\n"
        "Get a key at https://account.ipums.org/api_keys"
    )

client = IpumsApiClient(api_key)
print("API client ready.")

API client ready.


In [3]:
# Clear all queued extracts so they don't block our new request.
# The IPUMS API doesn't expose a DELETE endpoint via ipumspy,
# so we use the underlying session directly.
import requests

base = f"{client.base_url}/extracts"
params = {"collection": "nhgis", "version": client.api_version}

# List recent extracts
resp = client.session.get(base, params={**params, "pageSize": 20})
extracts = resp.json().get("data", [])

deleted = []
for ext in extracts:
    if ext.get("status") == "queued":
        eid = ext["number"]
        r = client.session.delete(f"{base}/{eid}", params=params)
        if r.status_code < 300:
            deleted.append(eid)
            print(f"  Deleted extract #{eid}")
        else:
            print(f"  Could not delete extract #{eid} (HTTP {r.status_code}: {r.text[:100]})")

if deleted:
    print(f"\nCleared {len(deleted)} queued extract(s): {deleted}")
else:
    print("No queued extracts to clear.")

No queued extracts to clear.


---
## 3. Inspect metadata for Time Series Table A00

Before submitting an extract we can query the API for metadata about
table A00 to confirm its description, available geographic levels, and
the census years it covers.

In [4]:
meta = client.get_metadata(TimeSeriesTableMetadata("nhgis", "A00"))

print(f"Table:        A00")
print(f"Description:  {meta.description}")
print(f"Integration:  {getattr(meta, 'geographic_integration', 'nominal')}")
print(f"Geog levels:  {meta.geog_levels}")

years = getattr(meta, "years", None) or []
if years:
    print(f"Years:        {years}")

Table:        A00
Description:  Total Population
Integration:  Nominal
Geog levels:  [{'name': 'nation', 'description': 'Nation', 'hasGeogExtentSelection': False, 'sequence': 1}, {'name': 'state', 'description': 'State', 'hasGeogExtentSelection': True, 'sequence': 4}, {'name': 'county', 'description': 'State--County', 'hasGeogExtentSelection': True, 'sequence': 25}]
Years:        [{'name': '1790', 'description': '1790', 'sequence': 1}, {'name': '1800', 'description': '1800', 'sequence': 2}, {'name': '1810', 'description': '1810', 'sequence': 3}, {'name': '1820', 'description': '1820', 'sequence': 4}, {'name': '1830', 'description': '1830', 'sequence': 5}, {'name': '1840', 'description': '1840', 'sequence': 6}, {'name': '1850', 'description': '1850', 'sequence': 7}, {'name': '1860', 'description': '1860', 'sequence': 8}, {'name': '1870', 'description': '1870', 'sequence': 12}, {'name': '1880', 'description': '1880', 'sequence': 22}, {'name': '1890', 'description': '1890', 'sequence': 29

---
## 4. Define and submit the NHGIS extract

We request time-series table **A00** at the **county** geographic level.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `collection` | `"nhgis"` | NHGIS aggregate data collection |
| `time_series_tables` | `TimeSeriesTable("A00", geog_levels=["county"])` | Total population, county level, all available years |
| `data_format` | `"csv_header"` | CSV with a descriptive second header row |
| `tst_layout` | `"time_by_column_layout"` | Each census year becomes its own column |

In [ ]:
# Use the already-completed county-level extract #6
# (County total population, time series A00 nominal, all years — 221 KB)
EXTRACT_ID = 6

extract_info = client.get_extract_by_id(extract_id=EXTRACT_ID, collection="nhgis")
print(f"Extract #{EXTRACT_ID} — status: {extract_info.status}")
print(f"Description: {extract_info.description}")

---
## 5. Wait for the extract to complete and download it

The API processes the extract on IPUMS servers. `wait_for_extract` polls
with exponential back-off (starting at 1 s, capping at 300 s) until the
extract is ready, then we download the ZIP archive.

In [ ]:
# Point to the already-downloaded extract in the extract/ directory
ZIP_PATH = Path("extract/nhgis0006_csv.zip")
print(f"Using: {ZIP_PATH}")

---
## 6. Load the CSV from the downloaded ZIP

NHGIS packages its data inside the ZIP under a `*_csv/` sub-folder.
Because we chose `csv_header`, the file has a descriptive second row
that we skip when reading into Pandas.

In [ ]:
def load_nhgis_csv(zip_path: Path) -> pd.DataFrame:
    """Read the first data CSV from an NHGIS ZIP archive."""
    with zipfile.ZipFile(zip_path, "r") as zf:
        csv_files = [
            name for name in zf.namelist()
            if name.lower().endswith(".csv") and "_csv/" in name.replace("\\", "/")
        ]
        if not csv_files:
            raise FileNotFoundError("No CSV found inside the NHGIS zip.")

        chosen = csv_files[0]

        with zf.open(chosen) as f:
            df = pd.read_csv(f, header=0, skiprows=[1], low_memory=False)

    return df


raw = load_nhgis_csv(ZIP_PATH)
print(f"Loaded {len(raw)} rows, {len(raw.columns)} columns")
raw.head()

---
## 7. Reshape into wide format: one row per county, census years as columns

NHGIS names its data columns with a table/series prefix followed by a
4-digit year (e.g. `A00AA1790`, `A00AA2020`). We rename those columns
to just the year string, keep the identifier columns (GISJOIN, state,
county name, etc.), and filter to *current* counties by requiring a
non-empty 2020 population value.

In [ ]:
YEAR_PATTERN = re.compile(r"^.+?(\d{4})$")

id_cols = []
year_rename = {}  # original column name -> year string

for col in raw.columns:
    m = YEAR_PATTERN.match(str(col).strip())
    if m and 1790 <= int(m.group(1)) <= 2100:
        year_rename[str(col).strip()] = m.group(1)
    else:
        id_cols.append(str(col).strip())

year_cols = sorted(year_rename.values(), key=int)
print(f"Identifier columns: {id_cols}")
print(f"Census year columns: {year_cols}")

In [ ]:
# Rename data columns from e.g. A00AA2020 -> 2020
wide = raw.rename(columns=year_rename)

# Keep only rows that have a value for the 2020 census
if "2020" in wide.columns:
    wide = wide[wide["2020"].notna() & (wide["2020"].astype(str).str.strip() != "")].copy()
    print(f"Filtered to {len(wide)} rows with non-null 2020 population")

# Convert year columns to numeric
for yr in year_cols:
    if yr in wide.columns:
        wide[yr] = pd.to_numeric(wide[yr], errors="coerce")

# Final table: identifiers + year columns
pop_data = wide[id_cols + [y for y in year_cols if y in wide.columns]].copy()
pop_data.head(10)

---
## 8. Export to CSV

In [ ]:
pop_data.to_csv(CSV_OUTPUT, index=False)
print(f"Saved {CSV_OUTPUT}  ({pop_data.shape[0]} rows x {pop_data.shape[1]} cols)")

---
## 9. Graphical summary: U.S. total population over time

We sum population across all counties for each census year and plot the
result as a time series. Missing values are ignored in the sum.

In [ ]:
present_years = [y for y in year_cols if y in pop_data.columns]
totals = pop_data[present_years].sum(skipna=True)
years_int = [int(y) for y in totals.index]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years_int, totals.values, marker="o", linewidth=2, markersize=5, color="#2563eb")
ax.fill_between(years_int, totals.values, alpha=0.1, color="#2563eb")
ax.set_title("U.S. Total Population by Decennial Census (NHGIS A00 — County Sum)", fontsize=14)
ax.set_xlabel("Census Year", fontsize=12)
ax.set_ylabel("Population", fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = DATA_DIR / "us_total_population_timeseries.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved figure to {fig_path}")

---
## 10. Bonus: top 10 most populous counties over time

In [ ]:
# Top 10 most populous counties (by 2020 population) over time
if "2020" in pop_data.columns and len(pop_data) > 1:
    top10 = pop_data.nlargest(10, "2020")
    
    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in top10.iterrows():
        label = row.get("NHGISCODE", row.get("GISJOIN", ""))
        # Try to build a readable label from state + county name columns
        for name_col in ["GEOGRAPHIC_AREA", "GEONAME", "COUNTY"]:
            if name_col in row.index and pd.notna(row[name_col]):
                label = str(row[name_col])
                break
        vals = row[present_years].values.astype(float)
        ax.plot(years_int, vals, marker="o", markersize=3, linewidth=1.5, label=label)
    
    ax.set_title("Top 10 Most Populous Counties Over Time (NHGIS A00)", fontsize=14)
    ax.set_xlabel("Census Year", fontsize=12)
    ax.set_ylabel("Population", fontsize=12)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    fig_path2 = DATA_DIR / "top10_counties_timeseries.png"
    fig.savefig(fig_path2, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to {fig_path2}")
else:
    print("County-level data not available — skipping top-10 chart.")

---

## Summary

| Output file | Description |
|---|---|
| `data/county_population_by_census_year.csv` | Wide table — one row per current county, columns for each decennial census |
| `data/us_total_population_timeseries.png` | Aggregate U.S. population time series |
| `data/top10_counties_timeseries.png` | Population trends for the 10 largest counties |

**Data source:** IPUMS NHGIS, University of Minnesota, [www.nhgis.org](https://www.nhgis.org/).

**Citation:** Steven Manson, Jonathan Schroeder, David Van Riper, Tracy Kugler, and Steven Ruggles.
IPUMS National Historical Geographic Information System: Version 18.0 [dataset].
Minneapolis, MN: IPUMS. http://doi.org/10.18128/D050.V18.0